In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the feedback coefficient $a$ of $y[n] = x[n] + a\,y[n-1]$. On the left
is the impulse response $h[n] = a^n$, inside the dashed envelope $\pm|a|^n$.
On the right is the filter's gain at every frequency, in decibels. Positive
$a$ boosts the low frequencies and negative $a$ boosts the high ones, more and
more sharply as $|a|$ approaches $1$. Past $|a| = 1$, the impulse response
runs off the plot.

In [ ]:
# hide
# autorun
A0 = 0.9                            # the chapter's stable example

F_S = 44100
N_IR = 40
NI = np.arange(N_IR)
YLIM = 2.6
F = np.linspace(0.0, F_S / 2, 600)
EJW = np.exp(-2j * np.pi * F / F_S)


def stems(n, v):
    """x and y arrays that draw stems from 0 to v at each n, as one trace."""
    xs = np.repeat(n.astype(float), 3)
    ys = np.column_stack([np.zeros(len(v)), v, np.full(len(v), np.nan)]).ravel()
    xs[2::3] = np.nan
    return xs, ys


def impulse_response(a, NI=NI):
    return a ** NI                  # 1, a, a^2, ...


def response_db(a, EJW=EJW):
    if abs(a) >= 1:                 # no frequency response: it never settles
        return np.full(len(EJW), np.nan)
    return -20 * np.log10(np.abs(1 - a * EJW))


def figure():
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)
    h = impulse_response(A0)
    env = np.abs(A0) ** NI
    fig.add_scatter(x=np.concatenate([NI, [np.nan], NI]),
                    y=np.concatenate([env, [np.nan], -env]), mode="lines",
                    line=dict(color=IRON, width=1.2, dash="dash"), row=1, col=1)
    xs, ys = stems(NI, h)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color=RED, width=1.6),
                    row=1, col=1)
    fig.add_scatter(x=NI, y=h, mode="markers", marker=dict(color=RED, size=7),
                    cliponaxis=True, row=1, col=1)
    fig.add_scatter(x=F / 1000, y=response_db(A0), mode="lines",
                    line=dict(color=RED, width=2.2), row=1, col=2)
    fig.add_annotation(x=F_S / 4000, y=16, text="", showarrow=False,
                       font=dict(size=13, color=RED), xref="x2", yref="y2")
    fig.update_xaxes(range=[-0.8, N_IR - 0.2], title_text="Sample index n",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-YLIM, YLIM], title_text="h[n]",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, F_S / 2000], title_text="Frequency (kHz)",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[-10, 42], title_text="Magnitude (dB)",
                     fixedrange=True, row=1, col=2)
    return fig


def controls(fig):
    a = widgets.FloatSlider(description="Feedback $a$", min=-1.2, max=1.2,
                            step=0.01, value=A0)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(a, NI=NI, stems=stems, impulse_response=impulse_response,
               response_db=response_db, F_S=F_S, readout=readout):
        h = impulse_response(a)
        env = np.abs(a) ** NI
        xs, ys = stems(NI, h)
        unstable = abs(a) >= 1
        with fig.batch_update():
            fig.data[0].y = np.concatenate([env, [np.nan], -env])
            fig.data[1].y = ys
            fig.data[2].y = h
            fig.data[3].y = response_db(a)
            fig.layout.annotations[0].text = (
                "no frequency response:<br>the output never settles"
                if unstable else "")
        if a == 0:
            text = "<i>a</i> = 0: no feedback at all, so <i>h</i> = &delta;"
        elif abs(a) < 1:
            n60 = int(np.ceil(np.log(1e-3) / np.log(abs(a))))
            text = (f"|<i>a</i>| &lt; 1: stable. |<i>h</i>[<i>n</i>]| falls below "
                    f"0.001 (&minus;60 dB) after {n60:,} samples "
                    f"({1000 * n60 / F_S:.2f} ms)")
        elif abs(a) == 1:
            text = "|<i>a</i>| = 1: the impulse response never decays"
        else:
            n1k = int(np.ceil(np.log(1e3) / np.log(abs(a))))
            text = (f"|<i>a</i>| &gt; 1: unstable. |<i>h</i>[<i>n</i>]| passes "
                    f"1,000 after {n1k:,} samples")
        if a < 0:
            text += " &nbsp;·&nbsp; <i>a</i> &lt; 0 flips the sign every sample"
        readout.value = f"<span style='font-size:0.9em'>{text}</span>"

    widgets.interactive_output(update, {"a": a})
    return widgets.VBox([a, readout])

icm_plotly.show(figure, controls)